In [0]:
%run /Workspace/Users/omars.soub@gmail.com/DataBricks_Baraa_sales_Project/Analysis/Detailed_Python_Analysis/Data_Transformation

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS bara_slaes_project.gold.evaluation;

In [0]:
from sklearn.ensemble import (
    AdaBoostRegressor,
    GradientBoostingRegressor,
    BaggingRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import SGDRegressor
#from xgboost.spark import SparkXGBRegressor

In [0]:
import numpy as np
import mlflow
import mlflow.spark
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor
)

target="Sales"
train_df,test_df=df_ready.randomSplit([0.75,0.25])

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator



mlflow.sklearn.autolog()
mlflow.set_registry_uri("databricks2")

#mlflow.set_experiment("1st databriscks")
results = []
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/bara_slaes_project/gold/evaluation"

with mlflow.start_run(run_name="Cross-Validation Experiment-DataBricks_Baraa_sales_Project"):

    models = {
    "LinearRegression": LinearRegression(labelCol=target, featuresCol="features"),
    "DecisionTree": DecisionTreeRegressor(labelCol=target, featuresCol="features"),
    "RandomForest": RandomForestRegressor(labelCol=target, featuresCol="features"),
    "GBTRegressor": GBTRegressor(labelCol=target, featuresCol="features"),
    "AdaBoostRegressor":AdaBoostRegressor()
    }
    rmse_evaluator = RegressionEvaluator(
        labelCol=target,
        predictionCol="prediction",
        metricName="rmse"
    )
    

    for model_name, model_instance in models.items():

       
        with mlflow.start_run(nested=True, run_name=model_name) :
            mlflow.log_param("model_type", model_name)
            paramGrid = ParamGridBuilder().build()
            cv = CrossValidator(
                    estimator=model_instance,              
                    estimatorParamMaps=paramGrid,
                    evaluator=rmse_evaluator,
                    numFolds=2,                      
                    seed=42,                          
                    parallelism=4,
                    collectSubModels=False           
                )
            
            model=cv.fit(train_df)
            predictions = model.transform(test_df)
            rmse = rmse_evaluator.evaluate(predictions)
            mlflow.log_metric("RMSE", rmse)
    